# 03 — Model Training & SHAP Explainability

## Purpose
Train a churn prediction model, validate it rigorously, and then open the black
box using SHAP — translating model internals into plain business language.

## Why XGBoost?
- Handles mixed feature types (numeric + one-hot categoricals) natively
- Robust to outliers and skewed distributions
- Fast on 200k+ rows
- Tree-based, so SHAP TreeExplainer is exact (not approximate)

## Key Results
| Metric                    | Value                          |
|---------------------------|--------------------------------|
| ROC AUC (holdout 20%)     | **0.84**                       |
| #1 SHAP driver            | `contract_type_month_to_month` |
| #2 SHAP driver            | `is_autopay`                   |
| #3 SHAP driver            | `internet_service_fiber`       |
| #4 SHAP driver            | `is_on_promo`                  |
| #5 SHAP driver            | `tenure_months`                |

An AUC of **0.84** means the model correctly ranks a randomly-selected churner
above a randomly-selected retained customer 84% of the time. A random model scores 0.50.

## Feature Engineering — Preventing Data Leakage

The following columns are **excluded** from the model feature set:

| Column                   | Why Excluded                                                         |
|--------------------------|----------------------------------------------------------------------|
| `churn_score`            | Synthetic target driver — direct leakage                            |
| `billing_risk_score`     | Deterministic function of contract + payment features (multicollinear) |
| `cltv`                   | Derived from `monthly_charges` × margin — dashboard use only        |
| `monthly_charges_billed` | Derived from `monthly_charges` × promo — leakage proxy              |
| `customer_id`            | Non-predictive identifier                                            |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

STEEL_BLUE_900 = "#003865"

df = pd.read_csv("../data/telco_churn_225k_v120.csv")

# Exclude leakage / non-feature columns
drop_cols = ["customer_id", "churn", "churn_score", "churn_prob",
             "cltv", "billing_risk_score", "monthly_charges_billed"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df["churn"]

# One-hot encode categoricals
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows")
print(f"Class balance (test): {y_test.mean():.1%} churn")

## Model Training

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric="logloss"
)
model.fit(X_train, y_train)
print("Training complete.")

## Model Evaluation — ROC AUC = 0.84

The ROC curve plots the trade-off between sensitivity (catching churners) and
specificity (not bothering loyal customers). An AUC of **0.84** is strong for a
churn use case — most real-world telco models sit in the 0.75–0.88 range.

The steep initial climb of the curve shows the model can identify the
highest-risk customers very efficiently before false-positive rate rises much.

In [ ]:
y_pred_prob = model.predict_proba(X_test)[:, 1]
auc_score   = roc_auc_score(y_test, y_pred_prob)
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

print(f"ROC AUC: {auc_score:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color=STEEL_BLUE_900, lw=2, label=f"XGBoost (AUC = {auc_score:.2f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random Baseline")
plt.title("Receiver Operating Characteristic (ROC) Curve", color=STEEL_BLUE_900, fontsize=14)
plt.xlabel("False Positive Rate", color=STEEL_BLUE_900)
plt.ylabel("True Positive Rate", color=STEEL_BLUE_900)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/Model_ROC_Curve.png", dpi=150)
plt.show()

## Confusion Matrix

Using a 0.5 decision threshold, the confusion matrix shows absolute prediction
counts across the four outcome classes. In a retention context, **false negatives**
(churners predicted as retained) are typically more costly than false positives —
calibrating the threshold lower can improve recall at the cost of precision.

In [ ]:
y_pred_class = (y_pred_prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Retained", "Churned"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix (threshold = 0.50)", color=STEEL_BLUE_900, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/Model_ConfusionMatrix.png", dpi=150)
plt.show()

## SHAP Explainability

SHAP (SHapley Additive exPlanations) assigns each feature a contribution score
for every individual prediction. Three views are shown:

1. **Summary / Beeswarm** — each dot is one customer; color = feature value (red = high, blue = low)
2. **Global Importance** — mean |SHAP value| across all test customers, ranked

### Reading the SHAP Results

| Rank | Feature                        | Business Interpretation                                    |
|------|--------------------------------|------------------------------------------------------------|
| 1    | `contract_type_month_to_month` | By far the biggest driver — no lock-in = flight risk       |
| 2    | `is_autopay`                   | Manual payers are significantly more likely to churn        |
| 3    | `internet_service_fiber`       | Fiber customers have more competitive alternatives          |
| 4    | `is_on_promo`                  | Promo customers are price-sensitive; churn when offer ends  |
| 5    | `tenure_months`                | Short-tenure customers are at highest risk (negative signal)|
| 6–7  | `first_60d_tech_call` / `billing_call` | Early friction signals — confirms the EDA findings  |

The SHAP results closely mirror the EDA, which is a good sign — it means the model
has learned the real business drivers and isn't picking up on spurious correlations.

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Beeswarm plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Beeswarm — Churn Drivers per Customer", color=STEEL_BLUE_900, fontsize=14)
plt.tight_layout()
plt.savefig("../reports/Model_SHAP_Beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Global importance — mean |SHAP|
mean_shap   = np.abs(shap_values).mean(axis=0)
shap_df     = pd.DataFrame({"feature": X_test.columns, "mean_shap": mean_shap})
shap_df     = shap_df.nlargest(15, "mean_shap").sort_values("mean_shap")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(shap_df["feature"], shap_df["mean_shap"], color="#0072CE")
ax.set_xlabel("Mean |SHAP value| (Average Impact on Model Output)")
ax.set_title("Top 15 Features by Mean |SHAP| — Global Importance",
             color=STEEL_BLUE_900, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/Model_SHAP_GlobalImportance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Full summary plot (density-style)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type="dot", show=False)
plt.title("SHAP Summary — Feature Impact Direction & Magnitude", color=STEEL_BLUE_900, fontsize=14)
plt.tight_layout()
plt.savefig("../reports/Model_SHAP_Summary.png", dpi=150, bbox_inches="tight")
plt.show()